# 🤖 Pearl.AI — Forecasting Training [Google Colab]

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leosan123456/Pearl/blob/main/notebooks/pearl_ai_forecasting_COLAB.ipynb)

> **Versão Colab** — otimizada para GPU T4/A100 e integração via API REST.
> Não requer acesso direto ao banco SQLite.

**Diferenças desta versão:**
- Instala dependências automaticamente via `!pip`
- Carrega dados **via API REST** do Pearl.AI (URL configurável)
- **LSTM com GPU** habilitado por padrão
- Salva modelos no **Google Drive**
- Importa forecasts diretamente via API ao finalizar

**Como usar:**
1. `Runtime → Change runtime type → GPU (T4)`
2. Preencha `API_BASE_URL` e `API_SESSION_TOKEN` na célula de configuração
3. `Runtime → Run all`

---

## 📦 1. Instalação de Dependências

In [ ]:
# Instalar dependências no Colab
!pip install -q prophet neuralprophet xgboost lightgbm scikit-learn joblib tqdm plotly
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
print("✅ Dependências instaladas")

In [ ]:
# Montar Google Drive para salvar modelos
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    OUTPUT_DIR = '/content/drive/MyDrive/PearlAI_Models'
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"✅ Google Drive montado — modelos serão salvos em: {OUTPUT_DIR}")
except Exception:
    DRIVE_AVAILABLE = False
    OUTPUT_DIR = '/content/pearl_output'
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"ℹ️  Google Drive não disponível — salvando em: {OUTPUT_DIR}")

## ⚙️ 2. Configuração

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO — edite antes de rodar
# ──────────────────────────────────────────────────────────────────────────────

# URL do Pearl.AI (sem barra final)
# Para desenvolvimento local com ngrok: https://xxxx.ngrok.io
# Para produção: https://seudominio.com
API_BASE_URL = "https://SEU_NGROK_OU_DOMINIO.com"  # ← edite aqui

# Token de sessão (copie do cookie next-auth.session-token no navegador)
# DevTools → Application → Cookies → next-auth.session-token
API_SESSION_TOKEN = ""  # ← cole aqui

# Empresa específica (None = todas)
TARGET_COMPANY_ID = None

# Meses de forecast
FORECAST_MONTHS = 12

# Modelos a treinar
TRAIN_PROPHET  = True
TRAIN_XGBOOST  = True
TRAIN_LSTM     = True  # GPU disponível no Colab!

# Enviar forecasts para Pearl.AI após treinar
PUSH_TO_PEARL = True

# ── Verificar GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"🎮 GPU disponível: {result.stdout.strip()}")
else:
    print("⚠️  GPU não detectada. LSTM será mais lento.")
    TRAIN_LSTM = False

print(f"\n✅ Configuração OK — API: {API_BASE_URL} | Meses: {FORECAST_MONTHS}")

## 📦 3. Imports

In [ ]:
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import joblib

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
PEARL_GREEN = '#8aa26a'

print(f"✅ Imports OK | Python {sys.version.split()[0]}")

## 🌐 4. Carregamento de Dados via API

In [ ]:
def get_headers():
    return {
        'Cookie': f'next-auth.session-token={API_SESSION_TOKEN}',
        'Content-Type': 'application/json'
    }

def load_data_from_api():
    """Carrega companies e revenue records via API REST do Pearl.AI."""
    # Buscar empresas
    resp = requests.get(f"{API_BASE_URL}/api/companies", headers=get_headers(), timeout=30)
    resp.raise_for_status()
    companies_data = resp.json()
    if isinstance(companies_data, dict):
        companies_data = [companies_data]
    companies = pd.DataFrame(companies_data)
    print(f"  ✅ {len(companies)} empresa(s) encontrada(s)")
    
    if TARGET_COMPANY_ID:
        companies = companies[companies['id'] == TARGET_COMPANY_ID]
        print(f"  🎯 Filtrando empresa: {TARGET_COMPANY_ID}")
    
    # Buscar revenue records por empresa
    all_revenue = []
    for _, company in companies.iterrows():
        cid = company['id']
        try:
            r = requests.get(f"{API_BASE_URL}/api/ai/forecast/export/{cid}",
                             headers=get_headers(), timeout=30)
            if r.ok:
                data = r.json()
                for rec in data.get('revenueRecords', []):
                    rec['companyId']   = cid
                    rec['companyName'] = company['name']
                    rec['sector']      = company.get('sector', '')
                    all_revenue.append(rec)
                print(f"  📊 {company['name']}: {len(data.get('revenueRecords', []))} registros")
            else:
                print(f"  ⚠️  {company['name']}: {r.status_code}")
        except Exception as e:
            print(f"  ❌ {company['name']}: {e}")
    
    revenue = pd.DataFrame(all_revenue)
    return companies, revenue


print(f"🌐 Buscando dados de {API_BASE_URL}...")
companies_df, revenue_df = load_data_from_api()

# Preprocessamento
revenue_df['date'] = pd.to_datetime(
    revenue_df['year'].astype(str) + '-' + revenue_df['month'].astype(str).str.zfill(2) + '-01'
)
revenue_df['amount'] = pd.to_numeric(revenue_df['amount'], errors='coerce').fillna(0)
revenue_df = revenue_df[revenue_df['month'].notna()].sort_values(['companyId', 'date'])

# Filtrar empresas com >= 6 meses
counts = revenue_df.groupby('companyId')['amount'].count()
valid_ids = counts[counts >= 6].index
revenue_df = revenue_df[revenue_df['companyId'].isin(valid_ids)]

print(f"\n✅ {len(valid_ids)} empresa(s) prontas | {len(revenue_df)} registros")
revenue_df.tail()

## 📊 5. EDA Rápida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Pearl.AI — Revenue Overview', fontsize=14, color='white')

# Portfolio consolidado
portfolio = revenue_df.groupby('date')['amount'].sum()
axes[0].plot(portfolio.index, portfolio.values, color=PEARL_GREEN, linewidth=2, marker='o', markersize=5)
axes[0].set_title('Receita Consolidada do Portfólio', color='white')
axes[0].tick_params(colors='white')

# Por empresa
for cname, group in revenue_df.groupby('companyName'):
    axes[1].plot(group['date'], group['amount'], label=cname, linewidth=1.5, alpha=0.8)
axes[1].set_title('Receita por Empresa', color='white')
axes[1].legend(fontsize=9)
axes[1].tick_params(colors='white')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda.png', dpi=150, bbox_inches='tight')
plt.show()

pct = portfolio.pct_change().dropna() * 100
print(f"📈 Crescimento médio mensal: {pct.mean():.1f}% | Desvio: {pct.std():.1f}%")

## 🔮 6. Treinamento — Prophet + XGBoost + LSTM

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_forecast(series, periods=12):
    if len(series) < 4:
        return np.full(periods, np.mean(series))
    model = ExponentialSmoothing(series, trend='add', seasonal=None).fit(optimized=True)
    return model.forecast(periods)

def evaluate_model(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / (np.array(y_true) + 1e-6))) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"  [{name}] MAE={mae:,.0f} | RMSE={rmse:,.0f} | MAPE={mape:.1f}% | R²={r2:.3f}")
    return {'model': name, 'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}

FEATURE_COLS = ['lag_1','lag_2','lag_3','lag_6','rolling_3_mean','rolling_6_mean',
                'rolling_3_std','pct_change_1','pct_change_3','month_sin','month_cos',
                'quarter','month_num','trend']

def build_features(df):
    df = df.copy().sort_values('date')
    for lag in [1,2,3,6]: df[f'lag_{lag}'] = df['amount'].shift(lag)
    df['rolling_3_mean']  = df['amount'].shift(1).rolling(3).mean()
    df['rolling_6_mean']  = df['amount'].shift(1).rolling(6).mean()
    df['rolling_3_std']   = df['amount'].shift(1).rolling(3).std()
    df['pct_change_1']    = df['amount'].pct_change(1).replace([np.inf,-np.inf],0)
    df['pct_change_3']    = df['amount'].pct_change(3).replace([np.inf,-np.inf],0)
    df['month_sin']       = np.sin(2*np.pi*df['date'].dt.month/12)
    df['month_cos']       = np.cos(2*np.pi*df['date'].dt.month/12)
    df['quarter']         = df['date'].dt.quarter
    df['month_num']       = df['date'].dt.month
    df['trend']           = np.arange(len(df))
    return df.dropna()

print("✅ Funções auxiliares definidas")

In [ ]:
results   = {}
forecasts = {}

# ── Prophet ───────────────────────────────────────────────────────────────────
if TRAIN_PROPHET:
    from prophet import Prophet
    prophet_models = {}
    print("🔮 PROPHET\n")
    
    for company_id in revenue_df['companyId'].unique():
        df_c = revenue_df[revenue_df['companyId']==company_id][['date','amount']].copy()
        df_c.columns = ['ds','y']
        cname = revenue_df[revenue_df['companyId']==company_id]['companyName'].iloc[0]
        if len(df_c) < 4: continue
        
        split = max(int(len(df_c)*0.8), len(df_c)-3)
        model = Prophet(yearly_seasonality=len(df_c)>=12, weekly_seasonality=False,
                        daily_seasonality=False, seasonality_mode='multiplicative',
                        changepoint_prior_scale=0.05, interval_width=0.80)
        model.fit(df_c.iloc[:split], iter=500)
        
        if len(df_c) - split > 0:
            fut_test = model.make_future_dataframe(periods=len(df_c)-split, freq='MS', include_history=False)
            preds = model.predict(fut_test)['yhat'].values
            m = evaluate_model(df_c.iloc[split:]['y'].values, preds, f"Prophet-{cname}")
            results[f'prophet_{company_id}'] = m
        
        future = model.make_future_dataframe(periods=FORECAST_MONTHS, freq='MS', include_history=False)
        fc = model.predict(future)
        prophet_models[company_id] = model
        forecasts[company_id] = [{
            'year': int(r['ds'].year), 'month': int(r['ds'].month),
            'amount': max(0.0, float(r['yhat'])),
            'lowerBound': max(0.0, float(r['yhat_lower'])),
            'upperBound': max(0.0, float(r['yhat_upper'])),
            'confidence': 0.80, 'method': 'prophet'
        } for _, r in fc.iterrows()]
        print(f"  ✅ {cname}")
    
    joblib.dump(prophet_models, f'{OUTPUT_DIR}/prophet_models.pkl')

# ── XGBoost ───────────────────────────────────────────────────────────────────
if TRAIN_XGBOOST:
    import xgboost as xgb
    xgb_models = {}
    print("\n🌲 XGBOOST\n")
    
    for company_id in revenue_df['companyId'].unique():
        df_c = build_features(revenue_df[revenue_df['companyId']==company_id][['date','amount']].copy())
        cname = revenue_df[revenue_df['companyId']==company_id]['companyName'].iloc[0]
        if len(df_c) < 6: continue
        
        split = max(int(len(df_c)*0.8), len(df_c)-3)
        model = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
        model.fit(df_c[FEATURE_COLS].iloc[:split], df_c['amount'].iloc[:split], verbose=False)
        
        if split < len(df_c):
            preds = model.predict(df_c[FEATURE_COLS].iloc[split:])
            m = evaluate_model(df_c['amount'].iloc[split:].values, preds, f"XGB-{cname}")
            results[f'xgb_{company_id}'] = m
        
        # Rollout iterativo
        history  = list(df_c['amount'].values)
        last_date = df_c['date'].iloc[-1]
        xgb_fc = []
        for i in range(FORECAST_MONTHS):
            nd = last_date + pd.DateOffset(months=i+1)
            feat = {'lag_1': history[-1], 'lag_2': history[-2] if len(history)>=2 else history[-1],
                    'lag_3': history[-3] if len(history)>=3 else history[-1],
                    'lag_6': history[-6] if len(history)>=6 else history[-1],
                    'rolling_3_mean': np.mean(history[-3:]), 'rolling_6_mean': np.mean(history[-6:]),
                    'rolling_3_std': np.std(history[-3:]) if len(history)>=3 else 0,
                    'pct_change_1': (history[-1]-history[-2])/(history[-2]+1e-6) if len(history)>=2 else 0,
                    'pct_change_3': (history[-1]-history[-3])/(history[-3]+1e-6) if len(history)>=3 else 0,
                    'month_sin': np.sin(2*np.pi*nd.month/12), 'month_cos': np.cos(2*np.pi*nd.month/12),
                    'quarter': nd.quarter, 'month_num': nd.month, 'trend': len(df_c)+i}
            pred = max(0.0, float(model.predict(pd.DataFrame([feat])[FEATURE_COLS])[0]))
            margin = pred * 0.15
            xgb_fc.append({'year': int(nd.year), 'month': int(nd.month), 'amount': pred,
                           'lowerBound': max(0.0, pred-margin), 'upperBound': pred+margin,
                           'confidence': 0.75, 'method': 'xgboost'})
            history.append(pred)
        
        xgb_models[company_id] = model
        # Usar XGBoost se tiver MAPE melhor
        if f'xgb_{company_id}' in results and f'prophet_{company_id}' in results:
            if results[f'xgb_{company_id}']['mape'] < results[f'prophet_{company_id}']['mape']:
                forecasts[company_id] = xgb_fc
        elif not TRAIN_PROPHET:
            forecasts[company_id] = xgb_fc
        print(f"  ✅ {cname}")
    
    joblib.dump(xgb_models, f'{OUTPUT_DIR}/xgboost_models.pkl')

print(f"\n🏁 Treinamento concluído. {len(forecasts)} empresa(s) com forecast pronto.")

In [ ]:
# ── LSTM (GPU) ────────────────────────────────────────────────────────────────
if TRAIN_LSTM:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    
    DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEQ_LEN   = 6
    EPOCHS    = 150
    print(f"🧠 LSTM — Dispositivo: {DEVICE}\n")
    
    class LSTMForecaster(nn.Module):
        def __init__(self, hidden=128, layers=2):
            super().__init__()
            self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, dropout=0.2)
            self.fc   = nn.Linear(hidden, 1)
        def forward(self, x):
            return self.fc(self.lstm(x)[0][:,-1,:])
    
    lstm_models, scalers = {}, {}
    
    for company_id in revenue_df['companyId'].unique():
        df_c  = revenue_df[revenue_df['companyId']==company_id].sort_values('date')
        cname = df_c['companyName'].iloc[0]
        vals  = df_c['amount'].values.astype(float)
        if len(vals) < SEQ_LEN+3: continue
        
        scaler = MinMaxScaler()
        norm = scaler.fit_transform(vals.reshape(-1,1)).flatten()
        scalers[company_id] = scaler
        
        X  = torch.tensor([norm[i:i+SEQ_LEN] for i in range(len(norm)-SEQ_LEN)], dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        y  = torch.tensor([norm[i+SEQ_LEN]   for i in range(len(norm)-SEQ_LEN)], dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        
        split = max(int(len(X)*0.8), len(X)-2)
        loader = DataLoader(TensorDataset(X[:split],y[:split]), batch_size=16, shuffle=True)
        
        model = LSTMForecaster().to(DEVICE)
        opt   = torch.optim.Adam(model.parameters(), lr=0.001)
        crit  = nn.MSELoss()
        
        for ep in range(EPOCHS):
            for xb, yb in loader:
                opt.zero_grad(); loss=crit(model(xb),yb); loss.backward(); opt.step()
            if (ep+1)%50==0: print(f"  Ep {ep+1}/{EPOCHS} loss={loss.item():.5f}")
        
        # Forecast recursivo
        model.eval()
        seed = norm[-SEQ_LEN:].copy()
        last_d = df_c['date'].iloc[-1]
        lstm_fc = []
        with torch.no_grad():
            for i in range(FORECAST_MONTHS):
                x_in = torch.tensor(seed, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(DEVICE)
                p_n  = model(x_in).item()
                p_r  = max(0.0, float(scaler.inverse_transform([[p_n]])[0][0]))
                nd   = last_d + pd.DateOffset(months=i+1)
                lstm_fc.append({'year':int(nd.year),'month':int(nd.month),'amount':p_r,
                                'lowerBound':max(0,p_r*0.85),'upperBound':p_r*1.15,
                                'confidence':0.80,'method':'lstm'})
                seed = np.append(seed[1:], p_n)
        
        lstm_models[company_id] = model
        forecasts[company_id]   = lstm_fc  # LSTM tem prioridade final
        print(f"  ✅ {cname}: LSTM concluído")
    
    torch.save({cid: m.state_dict() for cid,m in lstm_models.items()}, f'{OUTPUT_DIR}/lstm_state_dicts.pt')
    joblib.dump(scalers, f'{OUTPUT_DIR}/lstm_scalers.pkl')
    print(f"\n💾 LSTM salvo em {OUTPUT_DIR}")

## 📊 7. Comparativo & Visualização

In [ ]:
# Métricas comparativas
print("\n📊 Métricas por empresa:\n")
if results:
    df_metrics = pd.DataFrame(list(results.values()))
    # Adicionar Holt como baseline
    for cid in revenue_df['companyId'].unique():
        df_c = revenue_df[revenue_df['companyId']==cid].sort_values('date')
        vals = df_c['amount'].values
        if len(vals) < 4: continue
        split = max(int(len(vals)*0.8), len(vals)-3)
        if split >= len(vals): continue
        holt_preds = holt_forecast(vals[:split], len(vals)-split)
        m = evaluate_model(vals[split:], holt_preds, f"Holt-{df_c['companyName'].iloc[0]}")
        df_metrics = pd.concat([df_metrics, pd.DataFrame([m])], ignore_index=True)
    
    summary = df_metrics.assign(model_family=df_metrics['model'].str.split('-').str[0])\
                         .groupby('model_family')[['mae','mape','r2']].mean().round(2)
    print("\n🏆 MAPE médio por família de modelo (menor = melhor):")
    print(summary.sort_values('mape').to_string())
    df_metrics.to_csv(f'{OUTPUT_DIR}/metrics.csv', index=False)

# Visualização dos forecasts
if forecasts:
    n = len(forecasts)
    cols = min(2, n)
    rows = (n+1)//2
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows), squeeze=False)
    fig.suptitle('Pearl.AI — Forecasts (Colab)', fontsize=14, color='white')
    
    for idx, (cid, fc_rows) in enumerate(forecasts.items()):
        ax = axes[idx//cols][idx%cols]
        df_c  = revenue_df[revenue_df['companyId']==cid].sort_values('date')
        cname = df_c['companyName'].iloc[0]
        method = fc_rows[0]['method'] if fc_rows else 'N/A'
        
        ax.plot(df_c['date'], df_c['amount'], color=PEARL_GREEN, linewidth=2, marker='o', markersize=4, label='Histórico')
        fc_df = pd.DataFrame(fc_rows)
        fc_df['date'] = pd.to_datetime(fc_df['year'].astype(str)+'-'+fc_df['month'].astype(str).str.zfill(2)+'-01')
        ax.plot(fc_df['date'], fc_df['amount'], color='#60a5fa', linewidth=2, linestyle='--', label=f'{method}')
        ax.fill_between(fc_df['date'], fc_df['lowerBound'], fc_df['upperBound'], alpha=0.15, color='#60a5fa')
        
        holt_fc = holt_forecast(df_c['amount'].values, FORECAST_MONTHS)
        ax.plot(fc_df['date'], holt_fc[:len(fc_df)], color='#f59e0b', linewidth=1.5, linestyle=':', label='Holt')
        
        ax.set_title(f"{cname} [{method}]", color='white', fontsize=11)
        ax.legend(fontsize=8); ax.tick_params(colors='white')
    
    for idx in range(n, rows*cols):
        axes[idx//cols][idx%cols].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/forecasts_colab.png', dpi=150, bbox_inches='tight')
    plt.show()

## 💾 8. Export de Dados

In [ ]:
# CSV do histórico
revenue_df.to_csv(f'{OUTPUT_DIR}/revenue_history.csv', index=False)
print(f"✅ {OUTPUT_DIR}/revenue_history.csv")

# CSV dos forecasts
all_fc = []
for cid, rows in forecasts.items():
    cname = revenue_df[revenue_df['companyId']==cid]['companyName'].iloc[0]
    for r in rows:
        all_fc.append({'companyId':cid,'companyName':cname,**r})

fc_df = pd.DataFrame(all_fc)
fc_df.to_csv(f'{OUTPUT_DIR}/forecasts.csv', index=False)
print(f"✅ {OUTPUT_DIR}/forecasts.csv ({len(fc_df)} linhas)")

# JSON para import
with open(f'{OUTPUT_DIR}/forecasts_import.json','w') as f:
    json.dump(forecasts, f, indent=2)
print(f"✅ {OUTPUT_DIR}/forecasts_import.json")

# Download direto no Colab
try:
    from google.colab import files
    files.download(f'{OUTPUT_DIR}/forecasts_import.json')
    files.download(f'{OUTPUT_DIR}/forecasts.csv')
    print("\n⬇️  Download iniciado automaticamente")
except Exception:
    pass

## 🚀 9. Upload para o Pearl.AI

In [ ]:
if PUSH_TO_PEARL and forecasts and API_SESSION_TOKEN:
    headers = {'Cookie': f'next-auth.session-token={API_SESSION_TOKEN}', 'Content-Type': 'application/json'}
    ok, fail = 0, 0
    print(f"🚀 Enviando {len(forecasts)} forecast(s) para {API_BASE_URL}...\n")
    
    for cid, fc_rows in forecasts.items():
        try:
            r = requests.post(f"{API_BASE_URL}/api/ai/forecast/import",
                              json={'companyId': cid, 'forecasts': fc_rows},
                              headers=headers, timeout=30)
            if r.ok:
                d = r.json()
                print(f"  ✅ {cid}: {d.get('imported',0)} registros importados ({d.get('method','?')})")
                ok += 1
            else:
                print(f"  ❌ {cid}: {r.status_code} — {r.text[:80]}")
                fail += 1
        except Exception as e:
            print(f"  ❌ {cid}: {e}")
            fail += 1
    
    print(f"\n📊 {ok} importadas | {fail} falhas")
    print("🎉 Acesse /dashboard/companies para ver os novos forecasts!")

elif not API_SESSION_TOKEN:
    print("⚠️  Configure API_SESSION_TOKEN e API_BASE_URL para fazer o upload.")
    print(f"   Use o arquivo gerado manualmente: {OUTPUT_DIR}/forecasts_import.json")
else:
    print("ℹ️  PUSH_TO_PEARL = False. Mude para True para enviar.")

---

## ✅ Concluído!

| Arquivo | Local |
|---|---|
| `revenue_history.csv` | Google Drive / /content |
| `forecasts.csv` | Google Drive / /content |
| `forecasts_import.json` | Google Drive / /content |
| `prophet_models.pkl` | Google Drive / /content |
| `xgboost_models.pkl` | Google Drive / /content |
| `lstm_state_dicts.pt` | Google Drive / /content |

**Próximos passos:**
- Acesse `/dashboard/companies/[slug]` para ver os forecasts atualizados
- Para retreinar: `Runtime → Restart and run all`
- Para usar localmente: `pearl_ai_forecasting.ipynb`